# Retained-cycle observation budgets (E5.3)

**Purpose:** Evaluate fixed-model prediction sensitivity to five numbers of retained gait cycles.

**Required inputs:** Prepared Baček HDF5; EXT-S0 results ZIP; original reference ZIP; E3.4 results ZIP; exact approved method and E5.3 annex documents.

**Outputs:** Window manifests, 28,400 predictions and evaluation records.

**Usage:** Run after EXT-S0 and E3.4. Required source notebook filename: E5_3_Retained_Cycle_Budget_Local.ipynb. Files are read from the notebook’s current working directory unless specified in its configuration.


In [ ]:
# 1 — Immutable method identities, local sources and strict archive checks
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
import csv, hashlib, io, json, platform, sys, zipfile

import h5py
import numpy as np
import scipy
import sklearn
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

H5=Path('bacek_clean_common_angles_metabolic_dataset.h5')
EXT_ZIP=Path('EXT_S0_Shared_Preflight_results.zip')
REF_ZIP=Path('EXT_S0_Original_Reference_Records.zip')
E34_ZIP=Path('E3_4_Between_Cycle_Variability_results.zip')
APPROVED=Path('Third_FYP_E3_4_E5_3_Joint_Protocol_Amendment_v0.2_APPROVAL_READY.md')
ANNEX=Path('Third_FYP_E5_3_Retained_Cycle_Observation_Budget_Proposed_Experimental_Protocol_v0.2.md')
NOTEBOOK=Path('E5_3_Retained_Cycle_Budget_Local.ipynb')
OUT=Path('E5_3_Retained_Cycle_Budget_results')
ARCHIVE=Path('E5_3_Retained_Cycle_Budget_results.zip')

METHOD_ID='FYP-JA-2026-11-04-02'
SOURCE_SHA='c73ce561e94de52f1dc86128829c43ef536c4cf6b5f4aa9af1ea060409221fa3'
EXT_SHA='deb90024d9b67766161e651548b51d8a6ef99f104421155152ea9b0357ec2522'
REF_SHA='5fd8b80d99522f82c3d2f45000dc553edf9afe5b125ea862d7da9bb89dba33ce'
E34_SHA='1735285b42b9f0f767f941aad1a33501cce64f7d83b419254799465bd1b40604'
AMENDMENT_SHA='d9d0741420392592f30b79c7fe3812d7259b728b33914ad280227b73b5cbd1db'
ANNEX_SHA='48e11fdfd6db3a463919601ca395eec6c8ad6ecea1c8a70bb4841747707a00c2'
BOOTSTRAP_SHA='5d93bf4ef62f3d488243f0b1732c8260fde78e8e1e1dba7afe6fb79390660373'
CHANNELS=('hip_left','hip_right','knee_left','knee_right','ankle_left',
          'ankle_right','pelvis_sagittal','pelvis_frontal','pelvis_transverse')
BUDGETS=(5,10,20,50,100)
R=20
SEED=2026
ATOL_DEG=1e-10
PRED_ATOL=1e-12
METRIC_ATOL=1e-10
FOLD_SPEC={1:('D-ALL9',0.1,None),2:('D-ALL9',1.0,None),
           3:('P-ALL9',1000.0,10),4:('P-ALL9',1000.0,10),
           5:('P-ALL9',1000.0,10)}

checks=[]
def check(value,message):
    if not bool(value): raise AssertionError(message)
def verify(name,value):
    checks.append({'check':name,'status':'PASS' if bool(value) else 'FAIL'})
    check(value,name)
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()
def rows_csv(blob):return list(csv.DictReader(io.StringIO(blob.decode('utf-8-sig'))))
def save_csv(path,records,columns):
    with Path(path).open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=columns);w.writeheader();w.writerows(records)
def fmt(x):return format(float(x),'.17g')
def rmse(y,p):
    e=np.asarray(p,dtype=np.float64)-np.asarray(y,dtype=np.float64)
    return float(np.sqrt(np.mean(e**2,dtype=np.float64)))

for file in (H5,EXT_ZIP,REF_ZIP,E34_ZIP,APPROVED,ANNEX,NOTEBOOK):
    check(file.is_file(),f'Missing file: {file}')
for file, expected in ((H5,SOURCE_SHA),(EXT_ZIP,EXT_SHA),(REF_ZIP,REF_SHA),
                      (E34_ZIP,E34_SHA),(APPROVED,AMENDMENT_SHA),(ANNEX,ANNEX_SHA)):
    verify('file_sha256_'+file.name,sha(file)==expected)
check(not OUT.exists(),
      'E5.3 results folder already exists; archive/move it before rerunning to avoid mixing records.')
check(not ARCHIVE.exists(),'Existing E5.3 output ZIP; archive/move it before a new controlled run.')

with zipfile.ZipFile(EXT_ZIP) as z:
    names={Path(n).name:n for n in z.namelist() if not n.endswith('/')}
    ext_summary=json.loads(z.read(names['EXT_S0_summary.json']))
    ext_blobs={name:z.read(names[name]) for name in
        ('canonical_trial_ledger.csv','canonical_W_ALL9.csv',
         'canonical_D_ALL9.csv','canonical_original_cycle_indices.csv',
         'inner_fold_manifest.csv','outer_fold_manifest.csv')}
verify('accepted_EXT_S0_status',ext_summary['status']=='PASS' and
       ext_summary['approved_method_release']==METHOD_ID and
       ext_summary['source_hdf5_sha256']==SOURCE_SHA)
for name, blob in ext_blobs.items():
    verify('canonical_file_sha_'+name,
           hashlib.sha256(blob).hexdigest()==ext_summary['files_sha256'][name])

with zipfile.ZipFile(REF_ZIP) as z:
    original_summary=json.loads(z.read('04/run_summary.json'))
    original_preds=rows_csv(z.read('04/held_out_predictions.csv'))
    original_models=rows_csv(z.read('04/selected_models.csv'))
    ref_outer=z.read('04/outer_fold_manifest.csv')
    ref_inner=z.read('04/inner_fold_manifest.csv')
    bootstrap_blob=z.read('14/bootstrap_resampled_participants.csv')
verify('original_foundational_run',original_summary['status']=='PASS' and
       original_summary['source_sha256']==SOURCE_SHA)
verify('original_partition_files',ref_outer==ext_blobs['outer_fold_manifest.csv'] and
       ref_inner==ext_blobs['inner_fold_manifest.csv'])
verify('bootstrap_csv_sha',hashlib.sha256(bootstrap_blob).hexdigest()==BOOTSTRAP_SHA)
with zipfile.ZipFile(E34_ZIP) as z:
    e34_summary=json.loads(z.read('E3_4_run_summary.json'))
verify('E34_shared_source_identity',e34_summary['source_sha256']==SOURCE_SHA and
       e34_summary['accepted_EXT_S0_results_zip_sha256']==EXT_SHA and
       e34_summary['approved_method_release']==METHOD_ID)

print('Approved files, original model records, EXT-S0 and E3.4 provenance: PASS')
print('Python',sys.version.split()[0],' NumPy',np.__version__,
      ' SciPy',scipy.__version__,' scikit-learn',sklearn.__version__,' h5py',h5py.__version__)

## Source and numerical checks


In [ ]:
# 2 — Strict canonical records, trial IDs, exact participant partitions and G model settings
trial_rows=rows_csv(ext_blobs['canonical_trial_ledger.csv'])
W_rows=rows_csv(ext_blobs['canonical_W_ALL9.csv'])
D_rows=rows_csv(ext_blobs['canonical_D_ALL9.csv'])
idx_rows=rows_csv(ext_blobs['canonical_original_cycle_indices.csv'])
outer_rows=rows_csv(ref_outer)
inner_rows=rows_csv(ref_inner)
UIDS=[r['trial_uid'] for r in trial_rows]
verify('284_unique_trial_UIDs',len(UIDS)==284 and len(set(UIDS))==284)
verify('canonical_UID_order',UIDS==[r['trial_uid'] for r in W_rows]==
       [r['trial_uid'] for r in D_rows])
Wcols=list(W_rows[0])[1:];Dcols=list(D_rows[0])[1:]
verify('W_909_column_contract',Wcols==
       [f'{c}_phase_{p:03d}' for c in CHANNELS for p in range(101)])
verify('D_54_column_contract',Dcols==
       [f'{c}_{desc}' for c in CHANNELS for desc in
        ('q0','q1','q2','q3','phase_sd','rom')])
Wref=np.asarray([[float(r[c]) for c in Wcols] for r in W_rows],dtype=np.float64)
Dref=np.asarray([[float(r[c]) for c in Dcols] for r in D_rows],dtype=np.float64)
N=np.asarray([int(r['retained_cycles']) for r in trial_rows],dtype=int)
y=np.asarray([float(r['observed_W_per_kg']) for r in trial_rows],dtype=np.float64)
people=np.asarray([r['subject_id'] for r in trial_rows])
folds=np.asarray([int(r['outer_fold']) for r in trial_rows],dtype=int)
uid_index={uid:i for i,uid in enumerate(UIDS)}
verify('trial_cycle_population',int(N.sum())==60757 and N.min()==100 and N.max()<=348 and
       len(set(people))==21)
verify('finite_W_D_and_targets',np.isfinite(Wref).all() and
       np.isfinite(Dref).all() and np.isfinite(y).all() and (y>0).all())

def split_people(s):return set(x for x in s.split(';') if x)
outer={int(row['outer_fold']):row for row in outer_rows}
inner={(int(row['outer_fold']),int(row['inner_fold'])):row for row in inner_rows}
verify('five_outer_twenty_inner',len(outer)==5 and len(inner)==20)
all_people=set(people.tolist())
for f in range(1,6):
    row=outer[f]
    dev=split_people(row['development_participants'])
    test=split_people(row['test_participants'])
    verify(f'outer_group_separation_{f}',dev.isdisjoint(test) and dev|test==all_people and
           np.array_equal(folds==f,np.isin(people,list(test))) and
           sum(folds==f)==int(row['n_test_trials']))
    covered=set()
    for j in range(1,5):
        ir=inner[(f,j)];tr=split_people(ir['train_participants']);va=split_people(ir['validation_participants'])
        val=set(np.flatnonzero(np.isin(people,list(va))).tolist())
        verify(f'inner_partition_{f}_{j}',tr.isdisjoint(va) and tr|va==dev and
               split_people(ir['outer_test_participants'])==test and
               not(covered&val) and len(val)==int(ir['n_validation_trials']))
        covered|=val
    verify(f'inner_coverage_{f}',covered==set(np.flatnonzero(folds!=f).tolist()))

orig_G={r['trial_uid']:r for r in original_preds if r['procedure']=='G_SELECTED'}
verify('original_284_G_reference_UIDs',set(orig_G)==set(UIDS) and len(orig_G)==284)
archived_G=np.asarray([float(orig_G[uid]['predicted_W_per_kg']) for uid in UIDS],dtype=np.float64)
for i,uid in enumerate(UIDS):
    r=orig_G[uid]
    verify('original_target_and_fold_'+uid, int(r['outer_fold'])==folds[i] and
           float(r['observed_W_per_kg'])==y[i] and r['subject_id']==people[i])
for f in range(1,6):
    m=[r for r in original_models if r['procedure']=='G_SELECTED' and int(r['outer_fold'])==f]
    rep,alpha,k=FOLD_SPEC[f]
    verify(f'original_selected_G_configuration_{f}',len(m)==1 and
           m[0]['representation']==rep and float(m[0]['alpha'])==alpha and
           (int(m[0]['n_components']) if m[0]['n_components'] else None)==k)

original_indices={uid:[] for uid in UIDS}
for row in idx_rows:
    uid=row['trial_uid']
    check(uid in original_indices, 'Unexpected source index UID')
    original_indices[uid].append((int(row['retained_row_0based']),int(row['original_cycle_number'])))
verify('60757_original_index_records',len(idx_rows)==60757)
for uid,n in zip(UIDS,N):
    seq=original_indices[uid]
    verify('source_row_index_'+uid,len(seq)==n and
           all(seq[i][0]==i for i in range(n)) and
           all(seq[i+1][1]>seq[i][1] for i in range(n-1)))
    original_indices[uid]=np.asarray([v for _,v in seq],dtype=np.int64)
print('Exact cohort, source-index records, grouped folds and original G selection: PASS')

In [ ]:
# 3 — Independent W/D mathematics; uniform-start and nested-window synthetic pilot

def D_from_mean(m):
    # m is channels x 101; exactly the frozen six-per-channel descriptor ordering.
    d=[]
    for channel in m:
        for q in range(4):
            d.append(float(np.trapezoid(channel[25*q:25*q+26],dx=1.0)/25.0))
        d.append(float(np.std(channel,ddof=0)))
        d.append(float(np.max(channel)-np.min(channel)))
    return np.asarray(d,dtype=np.float64)

def D_independent(m):
    out=[]
    for j in range(m.shape[0]):
        a=m[j].astype(np.float64)
        for q in range(4):
            z=a[q*25:q*25+26]
            out.append(float((z[0]/2+np.sum(z[1:-1])+z[-1]/2)/25))
        avg=float(np.sum(a)/101)
        out.append(float(np.sqrt(np.sum((a-avg)**2)/101)))
        out.append(float(np.max(a)-np.min(a)))
    return np.array(out,dtype=np.float64)

def start_for(u,n,k):
    assert 0<=u<1 and n>=k
    return int(np.floor(float(u)*(int(n)-int(k)+1)))

synthetic=np.arange(9*101,dtype=np.float64).reshape(9,101)/11.0
verify('independent_synthetic_D',np.max(np.abs(D_from_mean(synthetic)-
                                                 D_independent(synthetic)))<=1e-12)
constructed=np.stack([synthetic+0.5*c for c in range(12)])
verify('synthetic_selected_cycle_mean',np.max(np.abs(constructed[2:7].mean(axis=0)-
                             (synthetic+0.5*np.mean(np.arange(2,7)))))<=1e-12)
cumul=np.concatenate([np.zeros((1,9,101),dtype=np.float64),
                       np.cumsum(constructed,axis=0,dtype=np.float64)],axis=0)
verify('synthetic_cumulative_window_arithmetic',
       max(float(np.max(np.abs((cumul[s+k]-cumul[s])/k-
                   constructed[s:s+k].mean(axis=0,dtype=np.float64))))
           for s,k in ((0,5),(2,6),(5,7),(0,12)))<=1e-12)
# Outcome-blind N2 pilot: duplicate identical fitted pipelines on constructed inputs.
Xpilot=np.stack([(synthetic.reshape(-1)+j*0.01) for j in range(14)])
ypilot=np.sin(np.arange(14,dtype=np.float64))
for pipeline in (Pipeline([('scale',StandardScaler()),('ridge',Ridge(alpha=1.0))]),
                 Pipeline([('scale',StandardScaler()),
                           ('pca',PCA(n_components=5,whiten=False,random_state=2026)),
                           ('ridge',Ridge(alpha=1000.0))])):
    a=pipeline.fit(Xpilot,ypilot).predict(Xpilot[:4])
    b=sklearn.base.clone(pipeline).fit(Xpilot,ypilot).predict(Xpilot[:4])
    verify('synthetic_deterministic_G_pipeline_'+str(len(pipeline.steps)),
           np.max(np.abs(a-b))<=PRED_ATOL)
checks_nesting=0
for n in range(100,349):
    for u in (0.0,np.nextafter(1.0,0.0),0.001,0.1,0.24999,0.5,0.75001,0.999):
        prev=None
        for k in BUDGETS:
            s=start_for(u,n,k)
            assert 0<=s<=n-k
            if prev is not None:
                sp,kp=prev
                assert s<=sp and sp+kp<=s+k, (n,u,k,kp)
            prev=(s,k)
            checks_nesting+=1
verify('synthetic_uniform_start_nesting',checks_nesting==249*8*5)
for n in (100,141,348):
    for k in BUDGETS:
        counts=np.zeros(n-k+1,dtype=np.int64)
        for j in range(5000):
            u=(j+0.5)/5000
            counts[start_for(u,n,k)]+=1
        verify(f'constructed_uniform_start_{n}_{k}',counts.max()-counts.min()<=1)
print('Independent mean/descriptor and uniform-start/nesting tests: PASS')

In [ ]:
# 4 — Scan corrected source locally, independently rebuild all 284 full W/D arrays
# Only trial-sized angle arrays are loaded at once. No raw source is distributed.
computed_W=[];computed_D=[];local_index_gaps=[]
with h5py.File(H5,'r') as h5:
    verify('source_subject_set',set(h5['subjects'])==all_people)
    for i,row in enumerate(trial_rows):
        uid=UIDS[i]
        trial=h5['subjects'][row['subject_id']]['trials'][row['trial_id']]
        for attr in ('valid','comparison_eligible','unrestricted_comparison_eligible'):
            check(attr in trial.attrs and np.asarray(trial.attrs[attr]).reshape(-1)[0]==1,
                  f'{uid}: invalid source eligibility {attr}')
        n=int(N[i]);m=[]
        for c in CHANNELS:
            check(c in trial,f'{uid}: missing {c}')
            a=np.asarray(trial[c][()],dtype=np.float64)
            check(a.shape==(n,101) and np.isfinite(a).all(),f'{uid}: bad shape/data {c}')
            m.append(a.mean(axis=0,dtype=np.float64))
        m=np.stack(m)
        w=m.reshape(909);d=D_from_mean(m)
        # Separate target and original-index identity checks.
        check('metcost_mean' in trial and 'original_cycle_number' in trial,
              f'{uid}: missing target/index')
        target=float(np.asarray(trial['metcost_mean'][()]).reshape(-1)[0])
        ind=np.asarray(trial['original_cycle_number'][()]).reshape(-1)
        check(np.isfinite(ind).all() and np.array_equal(ind,original_indices[uid]),
              f'{uid}: original-cycle ordering differs from EXT-S0')
        check(target==y[i],f'{uid}: metabolic target differs from EXT-S0')
        check(np.max(np.abs(w-Wref[i]))<=ATOL_DEG and
              np.max(np.abs(d-Dref[i]))<=ATOL_DEG and
              np.max(np.abs(d-D_independent(m)))<=ATOL_DEG,
              f'{uid}: canonical full-trial W/D disagreement')
        computed_W.append(w);computed_D.append(d)
        local_index_gaps.append(int(ind[-1]-ind[0]+1-n))
Wlocal=np.vstack(computed_W);Dlocal=np.vstack(computed_D)
verify('full_909_canonical_numeric_rebuild',np.max(np.abs(Wlocal-Wref))<=ATOL_DEG)
verify('full_54_canonical_numeric_rebuild',np.max(np.abs(Dlocal-Dref))<=ATOL_DEG)
verify('cross_experiment_shared_canonical_features',
       e34_summary['source_and_canonical_mean_D_equal']=='PASS' and
       e34_summary['E3_4_V1_final_gate']=='PENDING_INDEPENDENT_AUDIT' and
       np.max(np.abs(Wlocal-Wref))<=ATOL_DEG and
       np.max(np.abs(Dlocal-Dref))<=ATOL_DEG)
verify('whole_trial_index_gap_ledger',sum(x>0 for x in local_index_gaps)==11 and
       sum(local_index_gaps)==12)
print('Source-level full-trial W/D and E3.4/EXT-S0 shared equality: PASS')

## Fixed-model replay


In [ ]:
# 5 — Refit exact five original G models, verify full-input prediction replay

def fixed_G_pipeline(f):
    rep,alpha,ncomp=FOLD_SPEC[f]
    steps=[('scale',StandardScaler())]
    if rep=='P-ALL9':
        steps.append(('pca',PCA(n_components=ncomp,whiten=False,random_state=2026)))
    steps.append(('ridge',Ridge(alpha=alpha,fit_intercept=True)))
    return Pipeline(steps)

models={};repeat_models={};local_G=np.full(284,np.nan,dtype=np.float64)
model_rows=[]
for f in range(1,6):
    train=np.flatnonzero(folds!=f);test=np.flatnonzero(folds==f)
    rep,alpha,ncomp=FOLD_SPEC[f]
    X=Dlocal if rep=='D-ALL9' else Wlocal
    mdl=fixed_G_pipeline(f).fit(X[train],y[train])
    mdl_again=fixed_G_pipeline(f).fit(X[train],y[train])
    pred=np.asarray(mdl.predict(X[test]),dtype=np.float64)
    again=np.asarray(mdl_again.predict(X[test]),dtype=np.float64)
    old=archived_G[test]
    max_diff=float(np.max(np.abs(pred-old)))
    same_run=float(np.max(np.abs(pred-again)))
    verify(f'G_original_284_replay_outer_{f}',max_diff<=PRED_ATOL)
    verify(f'G_same_runtime_repeat_outer_{f}',same_run<=PRED_ATOL)
    local_G[test]=pred
    models[f]=mdl;repeat_models[f]=mdl_again
    def arr_sha(arr):
        return hashlib.sha256(np.asarray(arr,dtype=np.float64).tobytes()).hexdigest()
    fitted_fingerprint=hashlib.sha256()
    for _,step in mdl.steps:
        for name in ('mean_','scale_','components_','explained_variance_',
                     'coef_','intercept_'):
            if hasattr(step,name):
                value=np.asarray(getattr(step,name),dtype=np.float64)
                fitted_fingerprint.update(name.encode());fitted_fingerprint.update(value.tobytes())
    model_rows.append({'outer_fold':f,'representation':rep,'alpha':fmt(alpha),
                       'pca_components':ncomp if ncomp is not None else '',
                       'development_participants':outer[f]['development_participants'],
                       'n_development_trials':len(train),'n_heldout_trials':len(test),
                       'development_trial_UIDs_SHA256':hashlib.sha256(
                           '\n'.join(UIDS[i] for i in train).encode()).hexdigest(),
                       'fitted_transform_model_SHA256':fitted_fingerprint.hexdigest(),
                       'max_archived_replay_abs_difference_W_per_kg':fmt(max_diff),
                       'max_same_runtime_repeat_abs_difference_W_per_kg':fmt(same_run),
                       'replay_tolerance_W_per_kg':fmt(PRED_ATOL),
                       'status':'PASS'})
verify('all_284_original_G_predictions',np.isfinite(local_G).all() and
       np.max(np.abs(local_G-archived_G))<=PRED_ATOL)
print('All 284 original G predictions strictly replayed: PASS; max diff',
      fmt(np.max(np.abs(local_G-archived_G))),'W/kg')

## Retained-cycle window sampling


In [ ]:
# 6 — Create immutable u, window, location-coverage and original-gap manifests
verify('sorted_trial_uid_sampling_order',UIDS==sorted(UIDS,key=lambda uid:tuple(uid.split('_'))))
OUT.mkdir(parents=True,exist_ok=False)
rng=np.random.default_rng(SEED)
draw_rows=[];window_rows=[];coverage_rows=[]
window_by_uid=defaultdict(list)
for i,uid in enumerate(UIDS):
    n=int(N[i]);source_idx=original_indices[uid]
    starts_by_K={k:[] for k in BUDGETS}
    gap_by_K={k:0 for k in BUDGETS}
    for repeat in range(1,R+1):
        u=float(rng.random())
        draw_rows.append({'trial_uid':uid,'repeat_id':repeat,
                          'u_decimal':fmt(u),'u_hex':u.hex(),
                          'seed':SEED,'generator':'numpy.random.default_rng/PCG64',
                          'numpy_version':np.__version__,
                          'iteration_index':len(draw_rows)})
        previous=None
        for k in BUDGETS:
            s=start_for(u,n,k);e=s+k-1
            if previous is not None:
                old_s,old_e=previous
                check(s<=old_s<=old_e<=e,f'{uid} repetition {repeat} nesting error')
            previous=(s,e)
            first=int(source_idx[s]);last=int(source_idx[e])
            span=int(last-first+1);missing=span-k
            check(missing>=0,f'{uid}: invalid inclusive original-cycle span')
            count=n-k+1
            q=(s/(n-k)) if count>1 else None
            location=('degenerate' if q is None else
                      'Q1' if q<0.25 else 'Q2' if q<0.5 else
                      'Q3' if q<0.75 else 'Q4')
            record={'trial_uid':uid,'subject_id':people[i],'outer_fold':int(folds[i]),
                    'repeat_id':repeat,'K':k,'u_hex':u.hex(),'retained_cycles_N':n,
                    'K_over_N':fmt(k/n),'valid_start_count':count,
                    'start_row_0based':s,'end_row_0based_inclusive':e,
                    'first_original_cycle_number':first,'last_original_cycle_number':last,
                    'inclusive_original_index_span':span,'missing_original_cycle_count':missing,
                    'original_index_gap':int(missing>0),'location_bin':location}
            window_rows.append(record)
            window_by_uid[uid].append(record)
            starts_by_K[k].append(s)
            gap_by_K[k]+=int(missing>0)
    for k in BUDGETS:
        starts=starts_by_K[k];count=n-k+1
        bins=Counter(r['location_bin'] for r in window_by_uid[uid] if r['K']==k)
        coverage_rows.append({'trial_uid':uid,'subject_id':people[i],
                   'outer_fold':int(folds[i]),'K':k,'N':n,'K_over_N':fmt(k/n),
                   'valid_start_count':count,'n_repetitions':R,
                   'unique_selected_starts':len(set(starts)),
                   'duplicates':R-len(set(starts)),
                   'fraction_valid_starts_sampled':fmt(len(set(starts))/count),
                   'location_degenerate':bins['degenerate'],
                   'location_Q1':bins['Q1'],'location_Q2':bins['Q2'],
                   'location_Q3':bins['Q3'],'location_Q4':bins['Q4'],
                   'n_windows_with_source_index_gap':gap_by_K[k],
                   'K_equals_N':int(k==n)})
verify('exact_5680_draws',len(draw_rows)==284*R and
       len({(r['trial_uid'],r['repeat_id']) for r in draw_rows})==284*R)
verify('exact_28400_windows',len(window_rows)==284*R*len(BUDGETS) and
       len({(r['trial_uid'],r['K'],r['repeat_id']) for r in window_rows})==284*R*len(BUDGETS))
verify('exact_1420_coverage_rows',len(coverage_rows)==284*len(BUDGETS))
verify('all_u_hex_roundtrips',all(float.fromhex(r['u_hex'])==float(r['u_decimal'])
                              for r in draw_rows))
verify('all_window_starts_replay',all(r['start_row_0based']==
         start_for(float.fromhex(r['u_hex']),r['retained_cycles_N'],r['K'])
         for r in window_rows))
verify('all_window_inclusive_gap_definitions',all(
       r['missing_original_cycle_count']==r['inclusive_original_index_span']-r['K']
       for r in window_rows))
verify('no_outcome_columns_in_window_manifest',all(not any('target' in k or
       'prediction' in k or 'error' in k for k in r) for r in window_rows))

sampling_summaries=[]
for k in BUDGETS:
    part=[r for r in coverage_rows if r['K']==k]
    fractions=np.array([float(r['K_over_N']) for r in part])
    starts=np.array([r['valid_start_count'] for r in part],dtype=np.int64)
    cov=np.array([float(r['fraction_valid_starts_sampled']) for r in part])
    sampling_summaries.append({'K':k,'n_trials':len(part),
        'K_over_N_min':fmt(fractions.min()),
        'K_over_N_median':fmt(np.median(fractions)),
        'K_over_N_max':fmt(fractions.max()),
        'valid_starts_min':int(starts.min()),
        'valid_starts_median':fmt(np.median(starts)),
        'valid_starts_max':int(starts.max()),
        'selected_unique_start_fraction_mean':fmt(cov.mean()),
        'selected_unique_start_fraction_min':fmt(cov.min()),
        'selected_unique_start_fraction_max':fmt(cov.max()),
        'windows_with_original_cycle_gap':sum(r['n_windows_with_source_index_gap'] for r in part),
        'total_sampled_windows':len(part)*R,
        'n_location_degenerate':sum(r['location_degenerate'] for r in part)})
save_csv(OUT/'uniform_start_draw_manifest.csv',draw_rows,list(draw_rows[0]))
save_csv(OUT/'window_manifest.csv',window_rows,list(window_rows[0]))
save_csv(OUT/'window_coverage_diagnostics.csv',coverage_rows,list(coverage_rows[0]))
save_csv(OUT/'budget_sampling_coverage_summary.csv',sampling_summaries,list(sampling_summaries[0]))
print('Outcome-blind 5,680 draws and 28,400 window/gap records SAVED before scoring — PASS')

## Fixed-model evaluation


In [ ]:
# 7 — Score exactly the locked window manifest; strict N2 repeatability
reduced_rows=[];repeat_differences=[];subset_math_checks=[]
full_rows=[]
for i,uid in enumerate(UIDS):
    full_rows.append({'trial_uid':uid,'subject_id':people[i],
                      'outer_fold':int(folds[i]),'N':int(N[i]),
                      'observed_W_per_kg':fmt(y[i]),
                      'archived_G_prediction_W_per_kg':fmt(archived_G[i]),
                      'reconstructed_G_prediction_W_per_kg':fmt(local_G[i]),
                      'absolute_replay_difference_W_per_kg':fmt(abs(local_G[i]-archived_G[i]))})

with h5py.File(H5,'r') as h5:
    for i,uid in enumerate(UIDS):
        f=int(folds[i]); rep=FOLD_SPEC[f][0]; n=int(N[i])
        trial=h5['subjects'][trial_rows[i]['subject_id']]['trials'][trial_rows[i]['trial_id']]
        source=np.stack([np.asarray(trial[c][()],dtype=np.float64) for c in CHANNELS],axis=1)
        check(source.shape==(n,9,101),f'{uid}: source shape changed')
        # Cumulative-sum result uses 1 subtraction per window; cross-checked below.
        cumulative=np.concatenate([np.zeros((1,9,101),dtype=np.float64),
                                   np.cumsum(source,axis=0,dtype=np.float64)],axis=0)
        inputs=[]
        these=window_by_uid[uid]
        check(len(these)==R*len(BUDGETS),'Missing sampled windows for '+uid)
        for wr in these:
            k=int(wr['K']);s=int(wr['start_row_0based']);e=int(wr['end_row_0based_inclusive'])
            if k==n:
                m=Wlocal[i].reshape(9,101)
            else:
                m=(cumulative[e+1]-cumulative[s])/k
            check(np.isfinite(m).all(),f'{uid}: nonfinite selected mean')
            if wr['repeat_id'] in (1,20) and k in (5,100):
                direct=source[s:e+1].mean(axis=0,dtype=np.float64)
                difference=float(np.max(np.abs(m-direct)))
                check(difference<=ATOL_DEG,f'{uid}: cumulative/direct mean drift')
                subset_math_checks.append({'trial_uid':uid,'K':k,
                       'repeat_id':wr['repeat_id'],
                       'max_cumulative_vs_direct_mean_degrees':fmt(difference)})
            if rep=='D-ALL9':
                x=D_from_mean(m)
            else:
                x=m.reshape(909)
            inputs.append(x)
        Xsub=np.asarray(inputs,dtype=np.float64)
        check(Xsub.shape==(100,54 if rep=='D-ALL9' else 909),
              f'{uid}: reduced-feature dimension mismatch')
        pred=np.asarray(models[f].predict(Xsub),dtype=np.float64)
        repeat=np.asarray(repeat_models[f].predict(Xsub),dtype=np.float64)
        diff=float(np.max(np.abs(pred-repeat)))
        check(np.isfinite(pred).all() and np.isfinite(repeat).all() and diff<=PRED_ATOL,
              f'{uid}: N2 window prediction repeatability failed')
        repeat_differences.append({'trial_uid':uid,'outer_fold':f,
                                   'n_windows':len(these),
                                   'max_N2_abs_difference_W_per_kg':fmt(diff)})
        for j,wr in enumerate(these):
            p=float(pred[j]);
            if wr['K']==n:
                check(abs(p-local_G[i])<=PRED_ATOL,
                      f'{uid}: K=N must replay full prediction')
            reduced_rows.append({'trial_uid':uid,'subject_id':people[i],
                'outer_fold':f,'representation':rep,
                'repeat_id':int(wr['repeat_id']),'K':int(wr['K']),
                'start_row_0based':int(wr['start_row_0based']),
                'observed_W_per_kg':fmt(y[i]),
                'prediction_W_per_kg':fmt(p),
                'full_trial_prediction_W_per_kg':fmt(local_G[i])})
        if (i+1)%50==0:print('Scored',i+1,'of 284 trials')
verify('28400_reduced_G_predictions',len(reduced_rows)==28400 and
       len({(r['trial_uid'],r['K'],r['repeat_id']) for r in reduced_rows})==28400)
verify('N2_all_repeat_predictions',len(repeat_differences)==284 and
       max(float(r['max_N2_abs_difference_W_per_kg']) for r in repeat_differences)<=PRED_ATOL)
verify('subset_direct_mean_numerical_check',len(subset_math_checks)==284*4 and
       max(float(r['max_cumulative_vs_direct_mean_degrees']) for r in subset_math_checks)<=ATOL_DEG)
print('28,400 predictions completed; same-fit replay and no-refitting gates PASS')

## Error and uncertainty summaries


In [ ]:
# 8 — Pooled, within-trial, balanced, fold and split-half numerical summaries
idx_K={k:j for j,k in enumerate(BUDGETS)}
pred3=np.full((284,5,R),np.nan,dtype=np.float64)
for r in reduced_rows:
    ti=uid_index[r['trial_uid']];kj=idx_K[int(r['K'])];ri=int(r['repeat_id'])-1
    check(np.isnan(pred3[ti,kj,ri]),'Duplicate prediction identity')
    pred3[ti,kj,ri]=float(r['prediction_W_per_kg'])
verify('complete_284_5_20_tensor',np.isfinite(pred3).all())
err=pred3-y[:,None,None]
sh=pred3-local_G[:,None,None]
mean_sse=np.mean(err**2,axis=2,dtype=np.float64)
mean_shift2=np.mean(sh**2,axis=2,dtype=np.float64)
original_sse=(local_G-y)**2
base_rmse=float(np.sqrt(np.mean(original_sse,dtype=np.float64)))
y_mean=float(np.mean(y,dtype=np.float64))
y_var=float(np.mean((y-y_mean)**2,dtype=np.float64))
check(y_var>0,'Invalid observed target spread')
subj_ids=sorted(set(people.tolist()))
pid=np.asarray([subj_ids.index(s) for s in people],dtype=int)

def calc_metrics(sample_preds,sel=None):
    # sample_preds: n_trials x n_repetitions, includes the frozen trial target array.
    yy=y if sel is None else y[sel]
    pp=sample_preds if sel is None else sample_preds[sel]
    e=pp-yy[:,None]
    return float(np.sqrt(np.mean(e**2))),float(np.mean(np.abs(e))),float(np.mean(e))

metric_rows=[];participant_rows=[];fold_rows=[];trial_rows_diag=[];split_rows=[]
for j,k in enumerate(BUDGETS):
    S=float(np.sqrt(np.mean(mean_shift2[:,j],dtype=np.float64)))
    RM=float(np.sqrt(np.mean(mean_sse[:,j],dtype=np.float64)))
    mae=float(np.mean(np.abs(err[:,j,:]),dtype=np.float64))
    bias=float(np.mean(err[:,j,:],dtype=np.float64))
    r2=float(1-np.mean(mean_sse[:,j],dtype=np.float64)/y_var)
    grouped_mse=np.array([np.mean(mean_sse[pid==p,j]) for p in range(21)])
    balanced=float(np.sqrt(np.mean(grouped_mse)))
    metric_rows.append({'K':k,'n_trials':284,'n_repeats_per_trial':R,
        'full_input_RMSE_W_per_kg':fmt(base_rmse),'S_W_per_kg':fmt(S),
        'RMSE_K_W_per_kg':fmt(RM),'Delta_RMSE_K_minus_full_W_per_kg':fmt(RM-base_rmse),
        'MAE_W_per_kg':fmt(mae),'bias_W_per_kg':fmt(bias),'R2':fmt(r2),
        'participant_balanced_RMSE_W_per_kg':fmt(balanced)})
    for p,sid in enumerate(subj_ids):
        idx=np.flatnonzero(pid==p)
        participant_rows.append({'K':k,'subject_id':sid,'n_trials':len(idx),
           'RMSE_K_W_per_kg':fmt(np.sqrt(np.mean(mean_sse[idx,j]))),
           'full_RMSE_W_per_kg':fmt(np.sqrt(np.mean(original_sse[idx]))),
           'mean_signed_prediction_shift_W_per_kg':fmt(np.mean(sh[idx,j,:])),
           'S_W_per_kg':fmt(np.sqrt(np.mean(mean_shift2[idx,j])))})
    for f in range(1,6):
        idx=np.flatnonzero(folds==f)
        rk=float(np.sqrt(np.mean(mean_sse[idx,j])))
        rf=float(np.sqrt(np.mean(original_sse[idx])))
        fold_rows.append({'K':k,'outer_fold':f,'n_trials':len(idx),
            'selected_original_representation':FOLD_SPEC[f][0],
            'S_W_per_kg':fmt(np.sqrt(np.mean(mean_shift2[idx,j]))),
            'RMSE_K_W_per_kg':fmt(rk),'RMSE_full_W_per_kg':fmt(rf),
            'Delta_RMSE_K_minus_full_W_per_kg':fmt(rk-rf)})
    for t,uid in enumerate(UIDS):
        trial_rows_diag.append({'trial_uid':uid,'subject_id':people[t],
            'outer_fold':int(folds[t]),'K':k,'N':int(N[t]),
            'mean_signed_prediction_shift_W_per_kg':fmt(np.mean(sh[t,j,:])),
            'sd_prediction_shift_ddof1_W_per_kg':fmt(np.std(sh[t,j,:],ddof=1)),
            'mean_squared_prediction_shift_W2_per_kg2':fmt(mean_shift2[t,j]),
            'mean_squared_metabolic_error_W2_per_kg2':fmt(mean_sse[t,j])})
    halves=[]
    for label,slice_ in (('repetitions_1_to_10',slice(0,10)),
                          ('repetitions_11_to_20',slice(10,20))):
        s=float(np.sqrt(np.mean(sh[:,j,slice_]**2)))
        r=float(np.sqrt(np.mean(err[:,j,slice_]**2)))
        halves.append((s,r,r-base_rmse))
        split_rows.append({'K':k,'half':label,'n_trials':284,'n_repetitions':10,
                           'S_W_per_kg':fmt(s),'RMSE_K_W_per_kg':fmt(r),
                           'Delta_RMSE_W_per_kg':fmt(r-base_rmse),
                           'absolute_S_between_halves_difference_W_per_kg':'',
                           'absolute_RMSE_between_halves_difference_W_per_kg':'',
                           'absolute_Delta_between_halves_difference_W_per_kg':''})
    split_rows.append({'K':k,'half':'absolute_between_half_difference',
        'n_trials':284,'n_repetitions':10,
        'S_W_per_kg':'','RMSE_K_W_per_kg':'','Delta_RMSE_W_per_kg':'',
        'absolute_S_between_halves_difference_W_per_kg':fmt(abs(halves[0][0]-halves[1][0])),
        'absolute_RMSE_between_halves_difference_W_per_kg':fmt(abs(halves[0][1]-halves[1][1])),
        'absolute_Delta_between_halves_difference_W_per_kg':fmt(abs(halves[0][2]-halves[1][2]))})
verify('budget_and_diagnostic_counts',len(metric_rows)==5 and
       len(participant_rows)==105 and len(fold_rows)==25 and
       len(trial_rows_diag)==1420 and len(split_rows)==15)
print('Trial-weighted, participant-balanced, fold and split-half metrics computed')
for row in metric_rows:
    print('K=',row['K'],' S=',row['S_W_per_kg'],' RMSE=',row['RMSE_K_W_per_kg'])

In [ ]:
# 9 — Reuse the unchanged E5.2 10,000 participant-cluster draw schedule
bootstrap_rows=rows_csv(bootstrap_blob)
verify('bootstrap_exact_10000_draws',len(bootstrap_rows)==10000 and
       [int(r['draw_index']) for r in bootstrap_rows]==list(range(1,10001)))
boot_stats=[]
all_boot=np.empty((10000,5,3),dtype=np.float64)
for di,row in enumerate(bootstrap_rows):
    draw=row['population_sampled_participants'].split(';')
    check(len(draw)==21 and all(s in subj_ids for s in draw),
          f'Invalid E5.2 participant resample {di+1}')
    multiplicities=Counter(draw)
    weights=np.array([multiplicities[s] for s in people],dtype=np.float64)
    weight_sum=float(weights.sum())
    full_b=float(np.sqrt(np.dot(weights,original_sse)/weight_sum))
    for j,k in enumerate(BUDGETS):
        sb=float(np.sqrt(np.dot(weights,mean_shift2[:,j])/weight_sum))
        rb=float(np.sqrt(np.dot(weights,mean_sse[:,j])/weight_sum))
        db=rb-full_b
        all_boot[di,j,:]=(sb,rb,db)
        boot_stats.append({'draw_index':di+1,'K':k,'n_sampled_people':21,
              'n_effective_trial_rows':int(weight_sum),
              'S_W_per_kg':fmt(sb),'RMSE_K_W_per_kg':fmt(rb),
              'full_RMSE_W_per_kg':fmt(full_b),
              'Delta_RMSE_W_per_kg':fmt(db)})
verify('50000_bootstrap_statistics',len(boot_stats)==50000 and np.isfinite(all_boot).all())
boot_intervals=[]
for j,k in enumerate(BUDGETS):
    for t,(name,key) in enumerate((('S','S_W_per_kg'),('RMSE_K','RMSE_K_W_per_kg'),
                                    ('Delta_RMSE_K_minus_full','Delta_RMSE_K_minus_full_W_per_kg'))):
        qlo,qhi=np.percentile(all_boot[:,j,t],[2.5,97.5])
        boot_intervals.append({'K':k,'metric':name,'point_estimate_W_per_kg':
                               next(r[key] for r in metric_rows if r['K']==k),
                               'lower_2p5_W_per_kg':fmt(qlo),
                               'upper_97p5_W_per_kg':fmt(qhi),
                               'bootstrap_draws':10000,
                               'interpretation':'descriptive fixed-model participant-cluster pointwise'})
verify('15_descriptive_bootstrap_intervals',len(boot_intervals)==15)
print('Original 10,000 E5.2 participant-cluster resamples applied without refitting: PASS')

In [ ]:
# 10 — Independent arithmetic cross-checks, provenance and record integrity
# Reconstruct from the full-precision saved producer ledgers by an independent scalar loop.
for row in metric_rows:
    k=int(row['K']); records=[r for r in reduced_rows if int(r['K'])==k]
    verify('exact_5680_predictions_for_K_'+str(k),len(records)==284*R)
    sq_sh=sum((float(r['prediction_W_per_kg'])-
               float(r['full_trial_prediction_W_per_kg']))**2 for r in records)
    sq_err=sum((float(r['prediction_W_per_kg'])-
                float(r['observed_W_per_kg']))**2 for r in records)
    est_s=(sq_sh/len(records))**0.5;est_rm=(sq_err/len(records))**0.5
    verify('independent_S_reconstruction_K_'+str(k),
           abs(est_s-float(row['S_W_per_kg']))<=METRIC_ATOL)
    verify('independent_RMSE_reconstruction_K_'+str(k),
           abs(est_rm-float(row['RMSE_K_W_per_kg']))<=METRIC_ATOL)
    verify('independent_R2_reconstruction_K_'+str(k),
           abs((1-(sq_err/len(records))/y_var)-float(row['R2']))<=METRIC_ATOL)
verify('original_full_reference_RMSE_reconstruction',
       abs(rmse(y,local_G)-base_rmse)<=METRIC_ATOL)

# One independent participant-resampling spot check from saved data.
first=bootstrap_rows[0]['population_sampled_participants'].split(';')
w=Counter(first)
den=sum(w[s] for s in people)
for j,k in enumerate(BUDGETS):
    manualS=(sum(w[people[t]]*mean_shift2[t,j] for t in range(284))/den)**0.5
    manualRM=(sum(w[people[t]]*mean_sse[t,j] for t in range(284))/den)**0.5
    verify('bootstrap_first_draw_reconstruction_K_'+str(k),
           abs(manualS-all_boot[0,j,0])<=METRIC_ATOL and
           abs(manualRM-all_boot[0,j,1])<=METRIC_ATOL)

verify('matched_predictions_and_window_keys',
       {(r['trial_uid'],int(r['K']),int(r['repeat_id'])) for r in reduced_rows}==
       {(r['trial_uid'],int(r['K']),int(r['repeat_id'])) for r in window_rows})
print('Internal independent-formula verification: PASS; external N3 audit remains PENDING.')

## Export saved evidence


In [ ]:
# 11 — Save complete evidence ledgers and checksum-bound result archive
save_csv(OUT/'original_model_reconstruction.csv',model_rows,list(model_rows[0]))
save_csv(OUT/'eligible_trial_ledger.csv',trial_rows,list(trial_rows[0]))
save_csv(OUT/'full_trial_reference_predictions.csv',full_rows,list(full_rows[0]))
save_csv(OUT/'reduced_cycle_predictions.csv',reduced_rows,list(reduced_rows[0]))
save_csv(OUT/'budget_metrics.csv',metric_rows,list(metric_rows[0]))
save_csv(OUT/'monte_carlo_split_half_metrics.csv',split_rows,list(split_rows[0]))
save_csv(OUT/'budget_participant_diagnostics.csv',participant_rows,list(participant_rows[0]))
save_csv(OUT/'budget_fold_diagnostics.csv',fold_rows,list(fold_rows[0]))
save_csv(OUT/'budget_trial_shift_diagnostics.csv',trial_rows_diag,list(trial_rows_diag[0]))
save_csv(OUT/'budget_bootstrap_statistics.csv',boot_stats,list(boot_stats[0]))
save_csv(OUT/'budget_bootstrap_intervals.csv',boot_intervals,list(boot_intervals[0]))
save_csv(OUT/'source_subset_mean_crosschecks.csv',subset_math_checks,list(subset_math_checks[0]))
save_csv(OUT/'N2_window_prediction_repeatability.csv',repeat_differences,list(repeat_differences[0]))
save_csv(OUT/'verification_checks.csv',checks,list(checks[0]))
save_csv(OUT/'verification_issues.csv',[],['check','issue','status'])

notebook_doc=json.loads(NOTEBOOK.read_text(encoding='utf-8'))
notebook_code=''.join(''.join(c.get('source',[])) for c in notebook_doc['cells']
                      if c['cell_type']=='code')
from contextlib import redirect_stdout
with io.StringIO() as stream:
    with redirect_stdout(stream): np.show_config()
    numpy_build_config=stream.getvalue()
summary={'experiment':'E5.3','status':'PASS_LOCAL_PENDING_INDEPENDENT_N3',
    'approved_method_release':METHOD_ID,'source_sha256':SOURCE_SHA,
    'approved_amendment_sha256':AMENDMENT_SHA,'approved_E5_3_annex_sha256':ANNEX_SHA,
    'accepted_EXT_S0_results_zip_sha256':EXT_SHA,
    'original_reference_records_zip_sha256':REF_SHA,
    'verified_E3_4_results_zip_sha256':E34_SHA,
    'original_E5_2_bootstrap_csv_sha256':BOOTSTRAP_SHA,
    'notebook_file_sha256':sha(NOTEBOOK),
    'notebook_code_cell_sha256':hashlib.sha256(notebook_code.encode()).hexdigest(),
    'source_and_canonical_full_W_D_equal':'PASS',
    'exact_original_G_model_reconstruction_and_284_prediction_replay':'PASS',
    'N0_original_G_replay':'PASS','N2_fixed_model_window_repeatability':'PASS',
    'N3_external_independent_metric_reconstruction':'PENDING',
    'E5_3_V1_gate':'PENDING_INDEPENDENT_AUDIT',
    'gait_cycle_budgets':list(BUDGETS),'repetitions_per_trial':R,
    'RNG_seed':SEED,'sampling_rule':'uniform_valid_start_shared_u_nested',
    'draw_manifest_rows':len(draw_rows),'window_manifest_rows':len(window_rows),
    'reduced_prediction_rows':len(reduced_rows),'full_prediction_rows':len(full_rows),
    'unique_trial_targets':284,'participant_clusters':21,
    'original_G_replay_max_difference_W_per_kg':fmt(np.max(np.abs(local_G-archived_G))),
    'N2_prediction_repeatability_max_difference_W_per_kg':fmt(max(
        float(r['max_N2_abs_difference_W_per_kg']) for r in repeat_differences)),
    'source_original_index_gap_window_count':sum(r['original_index_gap'] for r in window_rows),
    'previously_inspected_outer_folds':True,
    'evidence_status':'secondary_exploratory_informed_internal_evaluation',
    'created_at_UTC':datetime.now(timezone.utc).isoformat(),
    'environment':{'python':sys.version.split()[0],'numpy':np.__version__,
        'scipy':scipy.__version__,'scikit_learn':sklearn.__version__,
        'h5py':h5py.__version__,'platform':platform.platform(),
        'numpy_blas_configuration':numpy_build_config},
    'tolerances':{'feature_atol_degrees':ATOL_DEG,
                  'N0_N2_prediction_atol_W_per_kg':PRED_ATOL,
                  'N3_metric_atol_W_per_kg':METRIC_ATOL}}
summary['files_sha256']={p.name:sha(p) for p in sorted(OUT.iterdir()) if p.is_file()}
(OUT/'E5_3_run_summary.json').write_text(json.dumps(summary,indent=2)+'\n',encoding='utf-8')
with zipfile.ZipFile(ARCHIVE,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file():z.write(p,arcname=p.name)
print('E5.3 local production completed; N0, N2 and internal checks passed.')
print('Return only',ARCHIVE,'for independent N3 and final E5.3-V1 assessment.')